# Unificação das bases Kaggle (outer merge)

Junta as 4 bases já tratadas em `tratadas/` por `cod_individuo`, preservando todas as linhas
(outer join, mesma abordagem da célula **TABELA OUTER** de `notebooks/Analise_exploratoria/Analise_exp.ipynb`).

- `cadastro_clientes_kaggle_tratada.csv` (coluna `id_cliente`)
- `contratos_apolices_kaggle_tratada.csv` (coluna `cod_individuo`, prefixo `IND-`)
- `Atendimento_Sinistros_Tratado_kaggle.csv` (coluna `cod_individuo`)
- `Engajamento_Marketing_Tratado_kaggle.csv` (coluna `cod_individuo`)


In [1]:
from functools import reduce

import pandas as pd

df_cadastro = pd.read_csv("tratadas/cadastro_clientes_kaggle_tratada.csv", dtype={"id_cliente": str})
df_contratos = pd.read_csv("tratadas/contratos_apolices_kaggle_tratada.csv", dtype={"cod_individuo": str})
df_sinistros = pd.read_csv("tratadas/Atendimento_Sinistros_Tratado_kaggle.csv", dtype={"cod_individuo": str})
df_mkt = pd.read_csv("tratadas/Engajamento_Marketing_Tratado_kaggle.csv", dtype={"cod_individuo": str})

for nome, df in [("cadastro", df_cadastro), ("contratos", df_contratos), ("sinistros", df_sinistros), ("mkt", df_mkt)]:
    print(f"{nome}: {df.shape}")


cadastro: (99000, 16)
contratos: (94000, 24)
sinistros: (94860, 14)
mkt: (95000, 26)


### Preparação antes do merge

- `Atendimento_Sinistros_Tratado_kaggle.csv` tem 1.728 linhas 100% duplicadas (mesmo `cod_individuo`
  e mesmos valores em todas as colunas) — removidas com `drop_duplicates()` para não multiplicar
  linhas no merge sem adicionar nenhuma informação nova.
- Todas as colunas de ID são renomeadas para `cod_individuo` e normalizadas (remove prefixo `IND-`
  e sufixo `.0`, se houver).

In [2]:
n_antes = len(df_sinistros)
df_sinistros = df_sinistros.drop_duplicates()
print(f"sinistros: {n_antes} -> {len(df_sinistros)} linhas ({n_antes - len(df_sinistros)} duplicadas removidas)")

dfs = [df_cadastro, df_contratos, df_sinistros, df_mkt]

for df in dfs:
    col_id = df.columns[0]
    df.rename(columns={col_id: "cod_individuo"}, inplace=True)
    df["cod_individuo"] = (
        df["cod_individuo"]
        .astype(str)
        .str.replace("IND-", "", regex=False)
        .str.replace(r"\.0$", "", regex=True)
        .str.strip()
    )

for df in dfs:
    print(df["cod_individuo"].duplicated().sum(), "chaves duplicadas restantes")


sinistros: 94860 -> 93132 linhas (1728 duplicadas removidas)
0 chaves duplicadas restantes
0 chaves duplicadas restantes
0 chaves duplicadas restantes
0 chaves duplicadas restantes


In [3]:
# Outer merge: preserva todas as linhas de todas as bases, mesmo sem correspondência nas outras
df_final = reduce(lambda left, right: pd.merge(left, right, on="cod_individuo", how="outer"), dfs)

print(f"Linhas: {len(df_final)} | Colunas: {df_final.shape[1]}")
print(f"IDs únicos esperados (união): {len(set(df_cadastro['cod_individuo']) | set(df_contratos['cod_individuo']) | set(df_sinistros['cod_individuo']) | set(df_mkt['cod_individuo']))}")
df_final.head()


Linhas: 100000 | Colunas: 77
IDs únicos esperados (união): 100000


,cod_individuo,idade,data_nascimento,genero,estado_civil,tem_filhos,qtd_dependentes,renda_anual,possui_imovel,valor_imovel,...,veiculo_Moto,veiculo_NaN,veiculo_Pickup,veiculo_SUV,veiculo_Van,regiao_Centro-Oeste,regiao_NaN,regiao_Nordeste,regiao_Sudeste,regiao_Sul
0,221300000002,53.0,14/06/1973,0.0,0.0,1.0,1.0,94000.0,1.0,185000.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,221300000020,52.0,14/06/1974,1.0,0.0,1.0,3.0,47000.0,1.0,305000.0,...,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
2,221300000097,61.0,16/06/1965,1.0,1.0,1.0,4.0,139100.0,1.0,381000.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
3,221300000148,56.0,15/06/1970,0.0,NaN,1.0,NaN,54100.0,1.0,274000.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
4,221300000166,28.0,08/06/1998,0.0,1.0,1.0,2.0,70100.0,1.0,307000.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [4]:
nome_arquivo = "Base_Unificada_Kaggle_Outer.csv"
df_final.to_csv(nome_arquivo, index=False)
print(f"Salvo em notebooks/bases/bases_kaggle/{nome_arquivo}")


Salvo em notebooks/bases/bases_kaggle/Base_Unificada_Kaggle_Outer.csv
